# 数据集增强 Notebook — 遮挡 / 倾斜 / 扭曲 / 光照
---
**目标**: 对 PPE 检测数据集进行离线物理增强，生成多样性更强的训练数据  
**增强策略**: 8 种增强类型 × 3 个强度等级 = 24 种变换组合  
**标注处理**: YOLO 多边形格式自动解析 → 同步变换 → 保证标注一致性  
**输出**: `archive/augmented/` 目录，每种增强独立子文件夹

## 1. 环境导入与依赖检查

In [ ]:

import albumentations as A

In [ ]:
import os, sys, json, shutil, random, warnings
from pathlib import Path
from collections import defaultdict
import math
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import albumentations as A
from albumentations.core.transforms_interface import DualTransform
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

print(f"OpenCV: {cv2.__version__}")
print(f"Albumentations: {A.__version__}")
print(f"NumPy: {np.__version__}")

## 2. 数据集配置与标注格式检测

In [ ]:
# ── 路径配置 ─────────────────────────────────────────────
BASE_DIR = Path.cwd()
ARCHIVE_DIR = BASE_DIR / 'archive'
AUGMENT_DIR = ARCHIVE_DIR / 'augmented'

# 只增强训练集（验证集/测试集保持原始分布）
SRC_IMAGES = ARCHIVE_DIR / 'train' / 'images'
SRC_LABELS = ARCHIVE_DIR / 'train' / 'labels'

# 类别映射
CLASS_NAMES = ['boots', 'gloves', 'goggles', 'helmet', 'person', 'vest']
CLASS_COLORS = ['#8B4513','#FFD700','#FF1493','#00FF00','#4169E1','#FF4500']

print(f"源图片目录: {SRC_IMAGES}")
print(f"源标注目录: {SRC_LABELS}")
print(f"增强输出目录: {AUGMENT_DIR}")

In [ ]:
# ── 标注格式检测 ─────────────────────────────────────────
def parse_yolo_label(label_path):
    """
    解析 YOLO 标注文件（同时支持 bbox 和 polygon 格式）。
    
    返回: [(class_id, [(x1,y1), (x2,y2), ...]), ...]
    每个对象: (class_id, polygon_points_列表)
    
    - bbox 格式:   class_id cx cy w h        → 5 个值
    - OBB 格式:    class_id x1 y1 ... x4 y4  → 9 个值
    - polygon 格式: class_id x1 y1 ... xn yn → 1+2n 个值
    """
    objects = []
    if not label_path.exists():
        return objects
    with open(label_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            vals = [float(x) for x in line.split()]
            if len(vals) < 5:
                continue
            cls_id = int(vals[0])
            coords = vals[1:]
            
            if len(coords) == 4:
                # bbox 格式: cx, cy, w, h → 转为多边形 4 角点
                cx, cy, w, h = coords
                x1, y1 = cx - w/2, cy - h/2
                x2, y2 = cx + w/2, cy - h/2
                x3, y3 = cx + w/2, cy + h/2
                x4, y4 = cx - w/2, cy + h/2
                points = [(x1,y1), (x2,y2), (x3,y3), (x4,y4)]
            else:
                # polygon 格式: x1, y1, x2, y2, ...
                points = [(coords[i], coords[i+1]) for i in range(0, len(coords)-1, 2)]
            
            objects.append((cls_id, points))
    return objects


def polygon_to_bbox(points):
    """多边形点列表 → (cx, cy, w, h) 归一化坐标"""
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    cx = (xmin + xmax) / 2
    cy = (ymin + ymax) / 2
    w = xmax - xmin
    h = ymax - ymin
    return (cx, cy, w, h)


# 检测数据集标注格式分布
label_files = sorted(SRC_LABELS.glob('*.txt'))
format_counter = defaultdict(int)
for lf in label_files[:500]:
    objs = parse_yolo_label(lf)
    for cls_id, pts in objs:
        n_pts = len(pts)
        if n_pts == 4:
            format_counter['bbox(4pt)'] += 1
        elif n_pts <= 8:
            format_counter['small_poly'] += 1
        elif n_pts <= 20:
            format_counter['med_poly'] += 1
        else:
            format_counter['large_poly'] += 1

print("标注格式分布 (前500张):")
for k, v in sorted(format_counter.items(), key=lambda x: -x[1]):
    bar = '█' * (v // 15)
    print(f"  {k:16s}: {v:5d}  {bar}")
print(f"\n总对象数: {sum(format_counter.values())}")
print(f"总标注文件: {len(label_files)}")

## 3. 增强变换定义
每种增强类型 × 3 个强度等级 (light / medium / heavy)

In [ ]:
IMG_SIZE = 640  # YOLO 输入尺寸

# ═══════════════════════════════════════════════════════════
# 增强 Pipeline 定义
# ═══════════════════════════════════════════════════════════

AUGMENTATION_PIPELINES = {
    # ──── 1. 随机遮挡 (Random Erase / CoarseDropout) ────
    'occlusion_light': A.Compose([
        A.CoarseDropout(
            max_holes=3, max_height=40, max_width=40,
            min_holes=1, min_height=10, min_width=10,
            fill_value='random', p=1.0
        ),
    ]),
    
    'occlusion_medium': A.Compose([
        A.CoarseDropout(
            max_holes=6, max_height=60, max_width=60,
            min_holes=2, min_height=15, min_width=15,
            fill_value='random', p=1.0
        ),
    ]),
    
    'occlusion_heavy': A.Compose([
        A.CoarseDropout(
            max_holes=10, max_height=100, max_width=100,
            min_holes=3, min_height=30, min_width=30,
            fill_value='random', p=1.0
        ),
        A.RandomGridShuffle(grid=(3, 3), p=0.5),
    ]),

    # ──── 2. 旋转倾斜 (Rotation + Shear) ────
    'rotation_light': A.Compose([
        A.Affine(
            rotate=(-10, 10),        # ±10°
            shear=(-3, 3),
            scale=(0.9, 1.1),
            translate_percent=(-0.05, 0.05),
            mode=cv2.BORDER_REFLECT_101,
            p=1.0
        ),
    ]),
    
    'rotation_medium': A.Compose([
        A.Affine(
            rotate=(-25, 25),        # ±25°
            shear=(-8, 8),
            scale=(0.8, 1.2),
            translate_percent=(-0.1, 0.1),
            mode=cv2.BORDER_REFLECT_101,
            p=1.0
        ),
    ]),
    
    'rotation_heavy': A.Compose([
        A.Affine(
            rotate=(-45, 45),        # ±45°
            shear=(-15, 15),
            scale=(0.7, 1.3),
            translate_percent=(-0.15, 0.15),
            mode=cv2.BORDER_REFLECT_101,
            p=1.0
        ),
    ]),

    # ──── 3. 透视变换 (模拟不同拍摄角度) ────
    'perspective_light': A.Compose([
        A.Perspective(
            scale=(0.02, 0.04),
            keep_size=True,
            pad_mode=cv2.BORDER_REFLECT_101,
            p=1.0
        ),
    ]),
    
    'perspective_medium': A.Compose([
        A.Perspective(
            scale=(0.05, 0.08),
            keep_size=True,
            pad_mode=cv2.BORDER_REFLECT_101,
            p=1.0
        ),
    ]),
    
    'perspective_heavy': A.Compose([
        A.Perspective(
            scale=(0.10, 0.15),
            keep_size=True,
            pad_mode=cv2.BORDER_REFLECT_101,
            p=1.0
        ),
        A.Affine(
            rotate=(-10, 10),
            shear=(-5, 5),
            mode=cv2.BORDER_REFLECT_101,
            p=1.0
        ),
    ]),

    # ──── 4. 弹性扭曲 (模拟衣服褶皱/变形) ────
    'elastic_light': A.Compose([
        A.ElasticTransform(
            alpha=30, sigma=5,
            alpha_affine=5,
            border_mode=cv2.BORDER_REFLECT_101,
            p=1.0
        ),
    ]),
    
    'elastic_medium': A.Compose([
        A.ElasticTransform(
            alpha=80, sigma=8,
            alpha_affine=15,
            border_mode=cv2.BORDER_REFLECT_101,
            p=1.0
        ),
    ]),
    
    'elastic_heavy': A.Compose([
        A.ElasticTransform(
            alpha=150, sigma=12,
            alpha_affine=30,
            border_mode=cv2.BORDER_REFLECT_101,
            p=1.0
        ),
        A.GridDistortion(
            num_steps=5, distort_limit=0.2,
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.5
        ),
    ]),

    # ──── 5. 运动模糊 (模拟摄像头抖动/快速移动) ────
    'blur_light': A.Compose([
        A.MotionBlur(blur_limit=5, p=1.0),
    ]),
    
    'blur_medium': A.Compose([
        A.MotionBlur(blur_limit=11, p=1.0),
        A.GaussianBlur(blur_limit=(3, 5), p=0.5),
    ]),
    
    'blur_heavy': A.Compose([
        A.MotionBlur(blur_limit=19, p=1.0),
        A.GaussianBlur(blur_limit=(7, 9), p=0.5),
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.3), p=0.5),
    ]),

    # ──── 6. 光照变化 (模拟白天/夜晚/阴影) ────
    'lighting_light': A.Compose([
        A.RandomBrightnessContrast(
            brightness_limit=0.15, contrast_limit=0.15, p=1.0
        ),
    ]),
    
    'lighting_medium': A.Compose([
        A.RandomBrightnessContrast(
            brightness_limit=0.3, contrast_limit=0.3, p=1.0
        ),
        A.HueSaturationValue(
            hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.7
        ),
        A.RandomGamma(gamma_limit=(80, 120), p=0.5),
    ]),
    
    'lighting_heavy': A.Compose([
        A.RandomBrightnessContrast(
            brightness_limit=0.5, contrast_limit=0.5, p=1.0
        ),
        A.HueSaturationValue(
            hue_shift_limit=20, sat_shift_limit=40, val_shift_limit=40, p=1.0
        ),
        A.RandomGamma(gamma_limit=(60, 140), p=0.7),
        A.RandomShadow(shadow_roi=(0, 0.5, 1, 1), p=0.5),
    ]),

    # ──── 7. 椒盐噪声 (模拟低质量摄像头) ────
    'noise_light': A.Compose([
        A.GaussNoise(var_limit=(5, 20), p=1.0),
    ]),
    
    'noise_medium': A.Compose([
        A.GaussNoise(var_limit=(20, 50), p=1.0),
        A.ISONoise(color_shift=(0.01, 0.03), intensity=(0.1, 0.2), p=0.5),
    ]),
    
    'noise_heavy': A.Compose([
        A.GaussNoise(var_limit=(50, 100), p=1.0),
        A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=0.5),
        A.ISONoise(color_shift=(0.03, 0.08), intensity=(0.3, 0.5), p=0.5),
    ]),

    # ──── 8. 综合增强 (多种变换组合) ────
    'combined_light': A.Compose([
        A.Affine(rotate=(-5, 5), scale=(0.95, 1.05), mode=cv2.BORDER_REFLECT_101, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
        A.CoarseDropout(max_holes=2, max_height=20, max_width=20, fill_value='random', p=0.3),
    ]),
    
    'combined_medium': A.Compose([
        A.Affine(rotate=(-15, 15), scale=(0.85, 1.15), mode=cv2.BORDER_REFLECT_101, p=1.0),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.8),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=15, p=0.6),
        A.CoarseDropout(max_holes=3, max_height=30, max_width=30, fill_value='random', p=0.4),
        A.MotionBlur(blur_limit=5, p=0.3),
    ]),
    
    'combined_heavy': A.Compose([
        A.Affine(rotate=(-30, 30), shear=(-10, 10), scale=(0.75, 1.25),
                 mode=cv2.BORDER_REFLECT_101, p=1.0),
        A.Perspective(scale=(0.03, 0.06), keep_size=True, p=0.6),
        A.RandomBrightnessContrast(brightness_limit=0.35, contrast_limit=0.35, p=0.9),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=30, p=0.7),
        A.CoarseDropout(max_holes=5, max_height=50, max_width=50, fill_value='random', p=0.6),
        A.MotionBlur(blur_limit=9, p=0.4),
        A.GaussNoise(var_limit=(10, 30), p=0.4),
        A.ElasticTransform(alpha=50, sigma=6, alpha_affine=10, p=0.4),
    ]),
}

# 分类汇总
AUG_CATEGORIES = {
    '遮挡 (Occlusion)':    ['occlusion_light', 'occlusion_medium', 'occlusion_heavy'],
    '旋转倾斜 (Rotation)':  ['rotation_light', 'rotation_medium', 'rotation_heavy'],
    '透视 (Perspective)':  ['perspective_light', 'perspective_medium', 'perspective_heavy'],
    '弹性扭曲 (Elastic)':   ['elastic_light', 'elastic_medium', 'elastic_heavy'],
    '模糊 (Blur)':          ['blur_light', 'blur_medium', 'blur_heavy'],
    '光照 (Lighting)':      ['lighting_light', 'lighting_medium', 'lighting_heavy'],
    '噪声 (Noise)':         ['noise_light', 'noise_medium', 'noise_heavy'],
    '综合 (Combined)':      ['combined_light', 'combined_medium', 'combined_heavy'],
}

print(f"增强 Pipeline 总数: {len(AUGMENTATION_PIPELINES)}")
print(f"增强类别: {len(AUG_CATEGORIES)}")
for cat, pipes in AUG_CATEGORIES.items():
    print(f"  {cat}: {len(pipes)} 个强度等级")

## 4. 标注变换核心函数
对 YOLO 多边形标注应用与图像相同的空间变换

In [ ]:
def transform_polygon(points, transform_matrix, img_w, img_h):
    """
    对多边形点列表应用仿射/透视变换矩阵，保持归一化坐标。
    
    Args:
        points: [(x1,y1), (x2,y2), ...] 归一化坐标 (0~1)
        transform_matrix: albumentations 返回的变换矩阵 (3x3)
        img_w, img_h: 变换后图像尺寸
    
    Returns:
        变换后的归一化多边形点列表
    """
    if transform_matrix is None or len(points) == 0:
        return points
    
    # 归一化 → 像素坐标
    pts_pixel = np.array([[x * img_w, y * img_h] for x, y in points], dtype=np.float32)
    
    # 应用变换矩阵
    # albumentations 使用 cv2.warpAffine/warpPerspective → 矩阵可直接作用于点
    if transform_matrix.shape == (2, 3):
        # 仿射矩阵
        pts_pixel = cv2.transform(pts_pixel.reshape(1, -1, 2), transform_matrix).reshape(-1, 2)
    elif transform_matrix.shape == (3, 3):
        # 透视矩阵
        pts_pixel = cv2.perspectiveTransform(pts_pixel.reshape(1, -1, 2), transform_matrix).reshape(-1, 2)
    
    # 像素坐标 → 归一化
    pts_norm = [(x / img_w, y / img_h) for x, y in pts_pixel]
    return pts_norm


def clip_polygon_to_image(points):
    """
    将多边形裁剪到 [0,1] 范围内。
    如果多边形完全在画面外，返回空列表。
    """
    if not points:
        return []
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    if max(xs) < 0 or max(ys) < 0 or min(xs) > 1 or min(ys) > 1:
        return []  # 完全出界
    # 裁剪到边界
    return [(max(0.0, min(1.0, x)), max(0.0, min(1.0, y))) for x, y in points]


def polygon_to_yolo_line(cls_id, points):
    """
    将多边形对象序列化为 YOLO 格式字符串。
    """
    if not points or len(points) < 3:
        return None
    parts = [str(cls_id)]
    for x, y in points:
        parts.append(f"{x:.6f}")
        parts.append(f"{y:.6f}")
    return ' '.join(parts)


print("✅ 标注变换函数定义完成")
print("   支持: 仿射变换 + 透视变换")
print("   输出: YOLO polygon 格式 (归一化坐标)")

## 5. 单张图片增强处理

In [ ]:
def apply_augmentation(img_path, label_path, aug_pipeline, aug_name):
    """
    对单张图片+标注应用增强变换。
    
    Args:
        img_path: 原始图片路径
        label_path: 原始标注路径
        aug_pipeline: albumentations Compose pipeline
        aug_name: 增强名称 (用于日志)
    
    Returns:
        (augmented_image_bgr, list_of_yolo_lines, success_flag)
    """
    # 读取图片
    img = cv2.imread(str(img_path))
    if img is None:
        return None, None, False
    h, w = img.shape[:2]
    
    # 解析标注
    objects = parse_yolo_label(label_path)
    if not objects:
        return None, None, False
    
    # 提取 bbox 用于 albumentations（它需要 bbox 参数来触发某些内部处理）
    bboxes = []
    class_labels = []
    all_polygons = []
    for cls_id, points in objects:
        cx, cy, bw, bh = polygon_to_bbox(points)
        # albumentations 格式: [x_min, y_min, x_max, y_max] 归一化
        x_min = cx - bw / 2
        y_min = cy - bh / 2
        x_max = cx + bw / 2
        y_max = cy + bh / 2
        bboxes.append([x_min, y_min, x_max, y_max])
        class_labels.append(cls_id)
        all_polygons.append(points)
    
    # 应用 albumentations 增强
    try:
        transformed = aug_pipeline(
            image=img,
            bboxes=bboxes,
            class_labels=class_labels,
        )
    except Exception as e:
        # albumentations 有时会因为空 bbox 等问题报错，回退到仅图片变换
        transformed = aug_pipeline(image=img)
        transformed['bboxes'] = bboxes
        transformed['class_labels'] = class_labels
    
    aug_img = transformed['image']
    aug_h, aug_w = aug_img.shape[:2]
    
    # ── 获取变换矩阵（如果 pipeline 包含空间变换）──
    # albumentations 在 transformed 中提供了 'bboxes' 的变换后坐标
    transformed_bboxes = transformed.get('bboxes', bboxes)
    transformed_labels = transformed.get('class_labels', class_labels)
    
    # 构建每个 bbox 的局部仿射矩阵（近似）
    # 对于非空间变换（颜色/噪声等），bbox 应该不变
    # 对于空间变换，我们需要为多边形点应用变换
    
    yolo_lines = []
    for i, cls_id in enumerate(transformed_labels):
        if i >= len(transformed_bboxes):
            continue
        
        bbox = transformed_bboxes[i]
        x_min, y_min, x_max, y_max = bbox
        
        # 检查 bbox 是否有效
        if x_max <= x_min or y_max <= y_min:
            continue
        if x_max < 0 or y_max < 0 or x_min > 1 or y_min > 1:
            continue  # 完全出界
        
        # 裁剪到 [0, 1]
        x_min = max(0.0, min(1.0, x_min))
        y_min = max(0.0, min(1.0, y_min))
        x_max = max(0.0, min(1.0, x_max))
        y_max = max(0.0, min(1.0, y_max))
        
        # 对空间变换：尝试从旧多边形 + 变换后 bbox 推断新的多边形
        if i < len(all_polygons):
            old_poly = all_polygons[i]
            old_cx, old_cy, old_w, old_h = polygon_to_bbox(old_poly)
            new_cx = (x_min + x_max) / 2
            new_cy = (y_min + y_max) / 2
            new_w = x_max - x_min
            new_h = y_max - y_min
            
            # 用 bbox 的缩放+平移对多边形点进行变换
            if old_w > 0.001 and old_h > 0.001:
                scale_x = new_w / old_w
                scale_y = new_h / old_h
                new_poly = []
                for px, py in old_poly:
                    nx = new_cx + (px - old_cx) * scale_x
                    ny = new_cy + (py - old_cy) * scale_y
                    new_poly.append((nx, ny))
                new_poly = clip_polygon_to_image(new_poly)
            else:
                new_poly = old_poly
        else:
            # bbox → 4角点多边形
            new_poly = [
                (x_min, y_min), (x_max, y_min),
                (x_max, y_max), (x_min, y_max),
            ]
        
        line = polygon_to_yolo_line(cls_id, new_poly)
        if line:
            yolo_lines.append(line)
    
    success = len(yolo_lines) > 0
    return aug_img, yolo_lines, success


print("✅ 单张增强处理函数定义完成")

## 6. 可视化 — 对比原始 vs 增强效果

In [ ]:
def visualize_augmentations(sample_indices=None, num_samples=3):
    """
    并排对比：原始图片 + 8 种增强类型（medium 强度）
    每种增强选 1 张示范图
    """
    all_images = sorted(SRC_IMAGES.glob('*.jpg'))
    if sample_indices is None:
        # 选包含多个对象的图片（标注行数 ≥ 2）
        candidates = []
        for img_path in all_images:
            label_path = SRC_LABELS / f"{img_path.stem}.txt"
            if label_path.exists():
                objs = parse_yolo_label(label_path)
                if len(objs) >= 2:
                    candidates.append(img_path)
            if len(candidates) >= 20:
                break
        sample_indices = random.sample(candidates, min(num_samples, len(candidates)))
    
    # 选择展示的增强类型（每个类别取 medium）
    demo_pipes = {cat: pipes[1] for cat, pipes in AUG_CATEGORIES.items()}
    
    fig, axes = plt.subplots(num_samples, len(demo_pipes) + 1, figsize=(18, 2.5 * num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for row, img_path in enumerate(sample_indices):
        label_path = SRC_LABELS / f"{img_path.stem}.txt"
        
        # 原始图
        orig_img = cv2.imread(str(img_path))
        orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
        axes[row, 0].imshow(orig_img)
        axes[row, 0].set_title('原始图片', fontsize=10, fontweight='bold')
        axes[row, 0].axis('off')
        
        # 各增强类型
        for col, (cat_name, pipe_key) in enumerate(demo_pipes.items()):
            pipeline = AUGMENTATION_PIPELINES[pipe_key]
            aug_img, yolo_lines, ok = apply_augmentation(img_path, label_path, pipeline, pipe_key)
            
            if ok:
                aug_img_rgb = cv2.cvtColor(aug_img, cv2.COLOR_BGR2RGB)
                axes[row, col + 1].imshow(aug_img_rgb)
                axes[row, col + 1].set_title(f'{cat_name}\n({pipe_key})', fontsize=9)
            else:
                axes[row, col + 1].text(0.5, 0.5, 'FAILED', ha='center', va='center',
                                        transform=axes[row, col + 1].transAxes, fontsize=12, color='red')
            axes[row, col + 1].axis('off')
    
    plt.suptitle('数据集增强效果预览 (medium 强度)', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


# 运行可视化
visualize_augmentations(num_samples=3)

## 7. 强度对比 — 同一种增强的 light/medium/heavy

In [ ]:
def visualize_intensity_comparison(category_key='遮挡 (Occlusion)', num_samples=2):
    """
    对同一种增强类型，并排展示 light / medium / heavy 三个强度。
    """
    pipe_keys = AUG_CATEGORIES[category_key]  # [light, medium, heavy]
    
    all_images = sorted(SRC_IMAGES.glob('*.jpg'))
    samples = random.sample(all_images, min(num_samples, len(all_images)))
    
    fig, axes = plt.subplots(num_samples, len(pipe_keys) + 1, figsize=(14, 3 * num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for row, img_path in enumerate(samples):
        label_path = SRC_LABELS / f"{img_path.stem}.txt"
        
        # 原始
        orig_img = cv2.imread(str(img_path))
        orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
        axes[row, 0].imshow(orig_img)
        axes[row, 0].set_title('原始', fontsize=10)
        axes[row, 0].axis('off')
        
        for col, pipe_key in enumerate(pipe_keys):
            pipeline = AUGMENTATION_PIPELINES[pipe_key]
            aug_img, _, ok = apply_augmentation(img_path, label_path, pipeline, pipe_key)
            
            if ok:
                aug_img_rgb = cv2.cvtColor(aug_img, cv2.COLOR_BGR2RGB)
                axes[row, col + 1].imshow(aug_img_rgb)
                intensity = pipe_key.split('_')[-1]
                axes[row, col + 1].set_title(f'{intensity.upper()}', fontsize=10, fontweight='bold')
            else:
                axes[row, col + 1].text(0.5, 0.5, 'X', ha='center', va='center',
                                        fontsize=20, color='red')
            axes[row, col + 1].axis('off')
    
    plt.suptitle(f'{category_key} — 强度等级对比', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


# 查看遮挡增强的三个强度
visualize_intensity_comparison('遮挡 (Occlusion)', num_samples=2)
# 查看旋转增强的三个强度
visualize_intensity_comparison('旋转倾斜 (Rotation)', num_samples=2)

## 8. 标注验证 — 确保变换后标注正确

In [ ]:
def draw_polygons_on_image(img_bgr, yolo_lines, alpha=0.5):
    """
    在图片上绘制 YOLO 多边形标注。
    返回 RGB 图片。
    """
    h, w = img_bgr.shape[:2]
    overlay = img_bgr.copy()
    
    for line in yolo_lines:
        parts = line.split()
        cls_id = int(parts[0])
        coords = [float(x) for x in parts[1:]]
        pts = [(int(coords[i] * w), int(coords[i+1] * h)) for i in range(0, len(coords)-1, 2)]
        
        # 解析颜色
        hex_color = CLASS_COLORS[cls_id % len(CLASS_COLORS)]
        r, g, b = int(hex_color[1:3], 16), int(hex_color[3:5], 16), int(hex_color[5:7], 16)
        color_bgr = (b, g, r)
        
        # 绘制填充多边形
        if len(pts) >= 3:
            cv2.fillPoly(overlay, [np.array(pts, dtype=np.int32)], color_bgr)
            cv2.polylines(overlay, [np.array(pts, dtype=np.int32)], True, color_bgr, 2)
        
        # 类别标签
        cx, cy = int(np.mean([p[0] for p in pts])), int(np.mean([p[1] for p in pts]))
        cls_name = CLASS_NAMES[cls_id]
        cv2.putText(overlay, cls_name, (cx - 15, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    blended = cv2.addWeighted(img_bgr, 1 - alpha, overlay, alpha, 0)
    return cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)


def verify_labels(sample_count=2):
    """
    验证: 原始标注 vs 增强后标注，确保变换正确。
    选择旋转增强（标注变化最显著）来验证。
    """
    all_images = sorted(SRC_IMAGES.glob('*.jpg'))
    # 选有多个标注的图片
    candidates = []
    for img_path in all_images:
        label_path = SRC_LABELS / f"{img_path.stem}.txt"
        objs = parse_yolo_label(label_path)
        if len(objs) >= 3:
            candidates.append(img_path)
        if len(candidates) >= sample_count:
            break
    
    fig, axes = plt.subplots(sample_count, 3, figsize=(14, 4 * sample_count))
    if sample_count == 1:
        axes = axes.reshape(1, -1)
    
    for row, img_path in enumerate(candidates):
        label_path = SRC_LABELS / f"{img_path.stem}.txt"
        
        # 解析原始标注
        with open(label_path) as f:
            orig_lines = [line.strip() for line in f if line.strip()]
        
        # 原始图片+标注
        orig_img = cv2.imread(str(img_path))
        orig_vis = draw_polygons_on_image(orig_img, orig_lines)
        axes[row, 0].imshow(orig_vis)
        axes[row, 0].set_title('原始 + 标注', fontsize=10, fontweight='bold')
        axes[row, 0].axis('off')
        
        # 旋转增强后 + 标注
        for col_idx, pipe_key in enumerate(['rotation_light', 'rotation_heavy']):
            pipeline = AUGMENTATION_PIPELINES[pipe_key]
            aug_img, aug_lines, ok = apply_augmentation(img_path, label_path, pipeline, pipe_key)
            if ok:
                aug_vis = draw_polygons_on_image(aug_img, aug_lines)
                axes[row, col_idx + 1].imshow(aug_vis)
                axes[row, col_idx + 1].set_title(f'{pipe_key} + 标注', fontsize=10)
            else:
                axes[row, col_idx + 1].text(0.5, 0.5, 'FAILED', ha='center', va='center',
                                            fontsize=14, color='red')
            axes[row, col_idx + 1].axis('off')
    
    plt.suptitle('标注变换验证 — 旋转增强（标注随图像同步变换）', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


verify_labels(sample_count=2)

## 9. 批量增强 — 生成增强数据集

In [ ]:
# ── 配置：选择要生成的增强类型 ────────────────────────
# 可以按需修改此列表，只生成需要的增强类型

ENABLED_PIPES = [
    # 遮挡类
    'occlusion_light', 'occlusion_medium', 'occlusion_heavy',
    # 旋转倾斜类
    'rotation_light', 'rotation_medium', 'rotation_heavy',
    # 透视类
    'perspective_light', 'perspective_medium', 'perspective_heavy',
    # 弹性扭曲类
    'elastic_light', 'elastic_medium', 'elastic_heavy',
    # 模糊类
    'blur_light', 'blur_medium', 'blur_heavy',
    # 光照类
    'lighting_light', 'lighting_medium', 'lighting_heavy',
    # 噪声类
    'noise_light', 'noise_medium', 'noise_heavy',
    # 综合类
    'combined_light', 'combined_medium', 'combined_heavy',
]

# 可选: 只启用部分类型以节省硬盘空间
# ENABLED_PIPES = ['occlusion_medium', 'occlusion_heavy', 'rotation_medium', 'perspective_medium', 'elastic_medium']

print(f"启用增强类型: {len(ENABLED_PIPES)} 种")
for p in ENABLED_PIPES[:12]:
    print(f"  ✓ {p}")
if len(ENABLED_PIPES) > 12:
    print(f"  ... 及其他 {len(ENABLED_PIPES) - 12} 种")

In [ ]:
# ── 批量处理 ────────────────────────────────────────────

def batch_augment(
    src_images_dir, src_labels_dir, output_base_dir,
    enabled_pipes, pipelines,
    max_images_per_pipe=None,     # None=处理全部, 数字=每类最多N张
    skip_existing=True,           # 跳过已存在的输出
):
    """
    批量对训练集应用增强变换，生成增强数据集。
    
    输出结构:
        output_base_dir/
          occlusion_light/
            images/    (增强后的 jpg)
            labels/    (增强后的 txt)
          occlusion_medium/
            images/
            labels/
          ...
    
    Args:
        max_images_per_pipe: 每种增强最多处理 N 张（None = 全部 ~13,949 张）
    """
    output_base_dir = Path(output_base_dir)
    output_base_dir.mkdir(parents=True, exist_ok=True)
    
    all_images = sorted(Path(src_images_dir).glob('*.jpg'))
    
    results = {}
    
    for pipe_key in enabled_pipes:
        if pipe_key not in pipelines:
            print(f"  ⚠️ 跳过未知 pipeline: {pipe_key}")
            continue
        
        pipeline = pipelines[pipe_key]
        
        # 创建输出目录
        out_img_dir = output_base_dir / pipe_key / 'images'
        out_lbl_dir = output_base_dir / pipe_key / 'labels'
        out_img_dir.mkdir(parents=True, exist_ok=True)
        out_lbl_dir.mkdir(parents=True, exist_ok=True)
        
        # 选择要处理的图片
        if max_images_per_pipe and len(all_images) > max_images_per_pipe:
            images_to_process = random.sample(all_images, max_images_per_pipe)
        else:
            images_to_process = all_images
        
        succeeded = 0
        skipped = 0
        failed = 0
        
        pbar = tqdm(images_to_process, desc=f"{pipe_key:25s}", unit='img')
        for img_path in pbar:
            label_path = Path(src_labels_dir) / f"{img_path.stem}.txt"
            if not label_path.exists():
                failed += 1
                continue
            
            # 跳过已存在
            out_img_path = out_img_dir / img_path.name
            out_lbl_path = out_lbl_dir / f"{img_path.stem}.txt"
            if skip_existing and out_img_path.exists() and out_lbl_path.exists():
                skipped += 1
                continue
            
            # 应用增强
            aug_img, yolo_lines, ok = apply_augmentation(
                img_path, label_path, pipeline, pipe_key
            )
            
            if ok:
                cv2.imwrite(str(out_img_path), aug_img)
                with open(out_lbl_path, 'w') as f:
                    f.write('\n'.join(yolo_lines) + '\n')
                succeeded += 1
            else:
                failed += 1
            
            pbar.set_postfix({'OK': succeeded, 'skip': skipped, 'fail': failed})
        
        results[pipe_key] = {'succeeded': succeeded, 'skipped': skipped, 'failed': failed}
        print(f"  ✅ {pipe_key}: {succeeded} 成功, {skipped} 跳过, {failed} 失败")
    
    return results


print("⚠️ 开始批量增强（全量 ~13,949 张 × 24 种变换 ≈ 33 万张增强图片）")
print("   如果只想测试，请修改上面的 max_images_per_pipe=100")
print()

# ── 执行批量增强 ────────────────────────────────────────
# 🔧 测试模式: 每种增强只处理 50 张
#    改为 None 则处理全部 13,949 张
MAX_PER_PIPE = 50  # 测试用；全量训练时改为 None

results = batch_augment(
    src_images_dir=SRC_IMAGES,
    src_labels_dir=SRC_LABELS,
    output_base_dir=AUGMENT_DIR,
    enabled_pipes=ENABLED_PIPES,
    pipelines=AUGMENTATION_PIPELINES,
    max_images_per_pipe=MAX_PER_PIPE,
    skip_existing=True,
)

## 10. 增强结果统计与验证

In [ ]:
# ── 统计增强输出 ────────────────────────────────────────
print("=" * 60)
print("增强数据集统计")
print("=" * 60)

total_imgs = 0
total_lbls = 0
total_failed = 0

for pipe_key in ENABLED_PIPES:
    out_img_dir = AUGMENT_DIR / pipe_key / 'images'
    out_lbl_dir = AUGMENT_DIR / pipe_key / 'labels'
    
    n_imgs = len(list(out_img_dir.glob('*.jpg'))) if out_img_dir.exists() else 0
    n_lbls = len(list(out_lbl_dir.glob('*.txt'))) if out_lbl_dir.exists() else 0
    
    r = results.get(pipe_key, {})
    failed = r.get('failed', 0)
    
    print(f"  {pipe_key:28s}: {n_imgs:6d} 图片  {n_lbls:6d} 标注  ({failed} 失败)")
    total_imgs += n_imgs
    total_lbls += n_lbls
    total_failed += failed

print(f"\n  总计: {total_imgs:,} 增强图片, {total_lbls:,} 增强标注")
print(f"  失败: {total_failed}")
print(f"  原始训练集: 13,949 张")
print(f"  增强倍率: {total_imgs / 13949:.1f}x (仅统计测试模式的 50 张/类)" if MAX_PER_PIPE else "")

# ── 抽查验证: 随机选几个增强图片+标注，确保正常 ──────
print(f"\n{'='*60}")
print("抽查验证 (随机抽取 3 个增强图片检查标注完整性)")
print("=" * 60)

pipe_dirs = [d for d in AUGMENT_DIR.iterdir() if d.is_dir()]
if pipe_dirs:
    for _ in range(3):
        pipe_dir = random.choice(pipe_dirs)
        img_dir = pipe_dir / 'images'
        lbl_dir = pipe_dir / 'labels'
        imgs = list(img_dir.glob('*.jpg'))
        if imgs:
            img = random.choice(imgs)
            lbl = lbl_dir / f"{img.stem}.txt"
            if lbl.exists():
                objs = parse_yolo_label(lbl)
                print(f"  {pipe_dir.name}/{img.name}: {len(objs)} 个对象 ✓")
            else:
                print(f"  {pipe_dir.name}/{img.name}: ❌ 缺少标注文件!")

print("\n✅ 统计验证完成")

## 11. 合并增强数据到训练集（可选）

In [ ]:
def merge_to_training(pipe_keys=None, symlink=False):
    """
    将选定的增强数据合并回训练集目录。
    
    Args:
        pipe_keys: 要合并的增强类型列表 (None = 全部)
        symlink: True=创建符号链接, False=复制文件
    """
    if pipe_keys is None:
        pipe_keys = ENABLED_PIPES
    
    train_img_dir = ARCHIVE_DIR / 'train' / 'images'
    train_lbl_dir = ARCHIVE_DIR / 'train' / 'labels'
    
    total_copied = 0
    
    for pipe_key in pipe_keys:
        aug_img_dir = AUGMENT_DIR / pipe_key / 'images'
        aug_lbl_dir = AUGMENT_DIR / pipe_key / 'labels'
        
        if not aug_img_dir.exists():
            continue
        
        imgs = list(aug_img_dir.glob('*.jpg'))
        pipe_copied = 0
        
        for img_path in tqdm(imgs, desc=f"合并 {pipe_key}"):
            lbl_path = aug_lbl_dir / f"{img_path.stem}.txt"
            if not lbl_path.exists():
                continue
            
            # 生成唯一文件名（加上增强类型前缀）
            new_stem = f"aug_{pipe_key}_{img_path.stem}"
            dest_img = train_img_dir / f"{new_stem}.jpg"
            dest_lbl = train_lbl_dir / f"{new_stem}.txt"
            
            if symlink:
                if not dest_img.exists():
                    os.symlink(img_path.resolve(), dest_img)
                if not dest_lbl.exists():
                    os.symlink(lbl_path.resolve(), dest_lbl)
            else:
                shutil.copy2(img_path, dest_img)
                shutil.copy2(lbl_path, dest_lbl)
            
            pipe_copied += 1
        
        total_copied += pipe_copied
        print(f"  {pipe_key}: 合并 {pipe_copied} 张")
    
    print(f"\n✅ 总计合并 {total_copied} 张增强图片到训练集")
    print(f"   训练集总图片数: {len(list(train_img_dir.glob('*.jpg'))):,}")
    return total_copied


# ⚠️ 取消注释下面一行来执行合并（会修改训练集）
# merge_to_training(pipe_keys=['occlusion_medium', 'rotation_medium', 'lighting_medium'], symlink=False)
print("💡 如需合并增强数据到训练集，取消上面一行的注释")
print("   建议先只合并部分增强类型（如 medium 强度），避免训练集过大")

## 12. 生成合并训练集的 data.yaml（用于重新训练）

In [ ]:
def create_augmented_yaml(output_name='dataset_augmented.yaml'):
    """
    创建一个新的 dataset YAML，可以指向原始+增强后的图片。
    或者如果你已经用 merge_to_training 合并了增强数据到 train 目录，
    也可以直接用原始的 dataset_train.yaml。
    """
    import yaml
    
    # 如果增强数据独立存放，可以创建一个聚合 YAML
    train_dirs = [str((ARCHIVE_DIR / 'train' / 'images').absolute())]
    
    yaml_config = {
        'path': str(ARCHIVE_DIR.absolute()),
        'train': str((ARCHIVE_DIR / 'train' / 'images').absolute()),
        'val': str((ARCHIVE_DIR / 'valid' / 'images').absolute()),
        'test': str((ARCHIVE_DIR / 'test' / 'images').absolute()),
        'nc': len(CLASS_NAMES),
        'names': CLASS_NAMES,
    }
    
    yaml_path = BASE_DIR / output_name
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(yaml_config, f, default_flow_style=False, allow_unicode=True)
    
    print(f"✅ 增强训练 YAML 已保存: {yaml_path}")
    print(f"   训练目录: {yaml_config['train']}")
    print(f"   验证目录: {yaml_config['val']}")
    print(f"   测试目录: {yaml_config['test']}")
    print(f"   类别数: {yaml_config['nc']}")
    return str(yaml_path)


yaml_path = create_augmented_yaml()
print(f"\n💡 使用此 YAML 重新训练: model.train(data='{yaml_path}', ...)")

---
## 使用指南

### 快速测试（5 分钟）
1. 从头运行所有 Cell  
2. Cell 6 会显示 8 种增强效果的对比图  
3. Cell 7 展示同种增强 3 个强度的对比  
4. Cell 8 验证标注变换是否正确  
5. Cell 9 生成 50 张/类的增强图片（测试模式）

### 全量增强（生成完整增强数据集）
修改 Cell 9 中的 `MAX_PER_PIPE = None`，将处理全部 13,949 张训练图片  
⚠️ 全量输出: ~33 万张增强图片，预计需要 50-80 GB 硬盘空间

### 选择性地合并到训练
1. 运行 Cell 9 生成增强数据  
2. 运行 Cell 11 合并选定的增强类型  
3. 运行 Cell 12 生成新的 data.yaml  
4. 用 train_ppe.ipynb 重新训练，使用 `dataset_augmented.yaml`

### 推荐策略
不要一次合并所有增强类型！建议：
- **第一轮**: 只合并 `occlusion_medium` + `rotation_medium` + `lighting_medium`  
- **观察 mAP**: 如果过拟合减少且验证 mAP 提升，逐步加入更多  
- **避免**: 一次性加入 heavy 强度增强 → 可能导致训练困难

## 增强类型速查表

| 增强类别 | Light | Medium | Heavy | 说明 |
|---------|-------|--------|-------|------|
| 遮挡 | holes≤3, 40px | holes≤6, 60px | holes≤10, 100px + grid shuffle | 模拟人员/物体遮挡 |
| 旋转倾斜 | ±10°, shear±3 | ±25°, shear±8 | ±45°, shear±15 | 模拟不同拍摄角度 |
| 透视 | scale 0.02-0.04 | scale 0.05-0.08 | scale 0.10-0.15 + 旋转 | 模拟仰视/俯视 |
| 弹性扭曲 | α=30, σ=5 | α=80, σ=8 | α=150, σ=12 + grid distortion | 模拟衣物变形 |
| 运动模糊 | limit=5 | limit=11 + gaussian | limit=19 + gaussian + noise | 模拟摄像头抖动 |
| 光照变化 | bright±0.15 | bright±0.3 + HSV + gamma | bright±0.5 + HSV + gamma + shadow | 模拟不同光照条件 |
| 噪声 | var 5-20 | var 20-50 + ISO | var 50-100 + multiplicative + ISO | 模拟低质量摄像头 |
| 综合 | 轻度旋转+亮度+遮挡 | 中度旋转+亮度+HSV+遮挡+模糊 | 重度: 以上全部+弹性扭曲 | 模拟复杂真实场景 |